# 05 - Train/Test Preparation and Model Development

**Phase 5: Train/Test Preparation** · Group AI-39

This notebook creates the single chronological train/test split used by all models, then trains the Phase 6 baseline and four classifiers. The final season (`2015/2016`) is held out as an untouched future test set. Historical-feature imputation values are calculated from training rows only.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import sklearn
print("scikit-learn version:", sklearn.__version__)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.train_models import (
    TEST_SEASON,
    tune_models,
    build_split_summary,
    chronological_split,
    load_features,
    summarize_class_distribution,
    build_training_summary,
    fit_models,
    save_predictions,
)

features_path = REPO_ROOT / "data" / "processed" / "features.csv"
features = load_features(features_path)


scikit-learn version: 1.9.1


In [2]:
prepared = chronological_split(features, test_season=TEST_SEASON)
tuning = tune_models(features)
tuning.tried_configs
split_summary = build_split_summary(prepared)
split_summary


,split,rows,season_start,season_end,date_start,date_end
0,train,19327,2009/2010,2014/2015,2009-07-11,2015-05-31
1,test,3326,2015/2016,2015/2016,2015-07-17,2016-05-25


In [3]:
class_distribution = pd.concat(
    [
        summarize_class_distribution(features["match_outcome"], "overall"),
        summarize_class_distribution(prepared.y_train, "train"),
        summarize_class_distribution(prepared.y_test, "test"),
    ],
    ignore_index=True,
)
class_distribution


,split,class,count,percentage
0,overall,Home Win,10351,45.6937
1,overall,Draw,5765,25.4492
2,overall,Away Win,6537,28.8571
3,train,Home Win,8892,46.0082
4,train,Draw,4910,25.4049
5,train,Away Win,5525,28.5870
6,test,Home Win,1459,43.8665
7,test,Draw,855,25.7066
8,test,Away Win,1012,30.4269


In [4]:
pd.Series(prepared.imputation_values, name="training_median")


Home_Form_Last_5            7.000000
Away_Form_Last_5            7.000000
Home_Avg_Goals_Last_5       1.200000
Away_Avg_Goals_Last_5       1.200000
Home_Avg_Conceded_Last_5    1.400000
Away_Avg_Conceded_Last_5    1.200000
Home_Win_Rate               0.454545
Away_Win_Rate               0.260870
Form_Diff                   0.000000
Abs_Form_Diff               3.000000
Name: training_median, dtype: float64

The model feature matrix contains historical numeric features and `league_id`. Identifiers, dates, season labels, and the target are retained only in the split row tables for traceability and are not passed as model features. No match IDs overlap between train and test, and the training period ends before the test period begins.

## Phase 6 - Model development

Every estimator below uses the same chronological split and feature columns. Hyperparameters are selected by macro F1 on validation season `2014/2015` using `2009/2010`–`2013/2014` as tuning training data. The final selected configurations are then refit on `2009/2010`–`2014/2015` and evaluated only on untouched `2015/2016`. Logistic Regression, Decision Tree, and Random Forest use `class_weight="balanced"`; Logistic Regression and KNN scale numeric historical features, while the tree models do not. `league_id` is one-hot encoded inside each model pipeline.

In [5]:
trained = fit_models(prepared, configs=tuning.selected_configs)
training_summary = build_training_summary(prepared, tuning.selected_configs)
training_summary


,model,training_rows,test_rows,feature_count_before_encoding,random_state,selected_config
0,majority_baseline,19327,3326,11,,{}
1,logistic_regression,19327,3326,11,42,"{'C': 1.0, 'class_weight': 'balanced'}"
2,decision_tree,19327,3326,11,42,"{'max_depth': 5, 'min_samples_leaf': 5, 'class..."
3,random_forest,19327,3326,11,42,"{'n_estimators': 300, 'max_depth': 8, 'min_sam..."
4,knn,19327,3326,11,42,{'n_neighbors': 11}


In [6]:
predictions_path = REPO_ROOT / "outputs" / "predictions" / "test_predictions.csv"
save_predictions(trained.predictions, predictions_path)
training_summary_path = REPO_ROOT / "outputs" / "tables" / "model_training_summary.csv"
training_summary.to_csv(training_summary_path, index=False)
print(predictions_path)
print(training_summary_path)


C:\Users\Shaza\Desktop\Y3S1\football-match-outcome-ml\outputs\predictions\test_predictions.csv
C:\Users\Shaza\Desktop\Y3S1\football-match-outcome-ml\outputs\tables\model_training_summary.csv


In [7]:
tuning_path = REPO_ROOT / "outputs" / "tables" / "hyperparameter_tuning.csv"
tuning.tried_configs.to_csv(tuning_path, index=False)
print(tuning_path)


C:\Users\Shaza\Desktop\Y3S1\football-match-outcome-ml\outputs\tables\hyperparameter_tuning.csv
